# Transit Route Generation Evaluation

This notebook evaluates two approaches for transit network design:
- **LC (Learning-based Construction)** – pure neural route generator
- **Neural BCO (Bee Colony Optimization with neural initialization)**

Tested on **Mandl's benchmark** and **Tartu real-world network**.

---

In [18]:
# import torch
# import numpy as np
# import matplotlib.pyplot as plt
# import networkx as nx
# from pathlib import Path
# import warnings
# warnings.filterwarnings("ignore")

# # =============================================================
# # 1. ROUTE SETS (твои тензоры)
# # =============================================================

# routes_no_penalty = [
#     torch.tensor([0, 1, 2, 5, 7, 9, 10]),
#     torch.tensor([11, 3, 5, 14, 6]),
#     torch.tensor([13, 12, 10]),
#     torch.tensor([8, 14]),
#     torch.tensor([7, 14]),
#     torch.tensor([4, 3])
# ]

# routes_with_penalty = [
#     torch.tensor([7, 5, 2, 1, 0]),
#     torch.tensor([8, 14, 6, 9, 10]),
#     torch.tensor([12, 10, 11]),
#     torch.tensor([4, 3, 5, 14]),
#     torch.tensor([14, 7]),
#     torch.tensor([12, 13])
# ]

# route_sets = [
#     ("Без detour penalty", routes_no_penalty),
#     ("С detour penalty", routes_with_penalty),
# ]


# # =============================================================
# # 2. LOAD MANDL INSTANCE (твоя функция)
# # =============================================================

# def load_mandl_instance():
#     ROOT_DIR = Path.cwd().parent.parent
#     DATA_DIR = ROOT_DIR / "examples/data"
#     data_dir = DATA_DIR / "banchmark"
#     instance_name = "Mandl"

#     coords_path = data_dir / f"{instance_name}Coords.txt"
#     node_locs = torch.tensor(np.genfromtxt(coords_path, skip_header=1), dtype=torch.float32)

#     tt_path = data_dir / f"{instance_name}TravelTimes.txt"
#     street_adj = torch.tensor(np.genfromtxt(tt_path), dtype=torch.float32)

#     dmd_path = data_dir / f"{instance_name}Demand.txt"
#     demand = torch.tensor(np.genfromtxt(dmd_path), dtype=torch.float32)

#     print(f"Mandl instance loaded:")
#     print(f"  Nodes: {node_locs.shape[0]}, Demand: {demand.shape}")

#     return {
#         'street_adj': street_adj,
#         'demand': demand,
#         'node_locs': node_locs
#     }


# mandl = load_mandl_instance()
# coords = mandl['node_locs'].numpy()
# travel_times = mandl['street_adj'].numpy()
# n_nodes = coords.shape[0]

# pos = {i: coords[i] for i in range(n_nodes)}


# # =============================================================
# # 3. BASE GRAPH
# # =============================================================

# G = nx.DiGraph()
# for i in range(n_nodes):
#     for j in range(n_nodes):
#         if i != j and np.isfinite(travel_times[i, j]):
#             G.add_edge(i, j, weight=travel_times[i, j])


# # =============================================================
# # 4. DRAW FIGURE (SIDE BY SIDE)
# # =============================================================

# fig, axes = plt.subplots(1, 2, figsize=(30, 12), constrained_layout=True)
# colors = plt.cm.Set1(np.linspace(0, 1, 10))
# node_size = 350
# offset_distance = 0.35


# for ax, (title, routes_tensor) in zip(axes, route_sets):

#     # ---- compute order for parallel edges ----
#     edge_order = {}
#     for idx, route in enumerate(routes_tensor):
#         r = route.tolist()
#         for k in range(len(r) - 1):
#             u, v = r[k], r[k+1]
#             edge_order.setdefault((u, v), []).append(idx)

#     # ---- background edges ----
#     for u, v in G.edges():
#         x1, y1 = pos[u]
#         x2, y2 = pos[v]
#         ax.plot([x1, x2], [y1, y2], color='lightgray', lw=1.2, alpha=0.6, zorder=1)

#     # ---- nodes ----
#     ax.scatter(coords[:, 0], coords[:, 1], c='black', s=node_size, zorder=5)
#     for i, (x, y) in enumerate(coords):
#         ax.text(x, y, str(i), fontsize=11, fontweight='bold',
#                 color='white', ha='center', va='center', zorder=6)

#     # ---- draw routes ----
#     handles = []

#     for idx, route in enumerate(routes_tensor):
#         r = route.tolist()
#         if len(r) < 2:
#             continue

#         color = colors[idx % len(colors)]
#         total_time = int(sum(travel_times[r[k], r[k+1]] for k in range(len(r) - 1)))

#         for k in range(len(r) - 1):
#             u, v = r[k], r[k+1]
#             x1, y1 = pos[u]
#             x2, y2 = pos[v]

#             # compute perpendicular shift
#             dx, dy = x2 - x1, y2 - y1
#             dist = np.hypot(dx, dy)
#             ux, uy = dx / dist, dy / dist
#             px, py = -uy, ux

#             order_list = edge_order[(u, v)]
#             pos_idx = order_list.index(idx)
#             offset = (pos_idx - (len(order_list) - 1) / 2) * offset_distance

#             ox, oy = px * offset, py * offset
#             x1o, y1o = x1 + ox, y1 + oy
#             x2o, y2o = x2 + ox, y2 + oy

#             ax.plot([x1o, x2o], [y1o, y2o], color=color, lw=4, alpha=0.9, zorder=4)
#             ax.annotate('', xy=(x2o, y2o), xytext=(x1o, y1o),
#                         arrowprops=dict(arrowstyle='->', lw=2.2, color=color),
#                         zorder=5)

#         # legend entry
#         label = f"Route {idx}: " + " → ".join(map(str, r)) + f"  ({total_time} min)"
#         handles.append(plt.Line2D([0], [0], color=color, lw=5, label=label))

#     # ---- style ----
#     ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.1),
#               ncol=1, fontsize=10)
#     ax.set_title(title, fontsize=20, fontweight='bold')
#     ax.set_aspect('equal')
#     ax.axis('off')

# # -------------------------------------------------------------
# plt.savefig("mandl_routes_no_penalty_vs_penalty.png", dpi=300, bbox_inches='tight')
# plt.show()


## 1. Setup & Configuration

In [19]:
import torch
import numpy as np
import pandas as pd
import networkx as nx
import pickle
from pathlib import Path
from omegaconf import OmegaConf
from torch_geometric.loader import DataLoader
from tqdm import tqdm

# Project-specific imports
from transport.routes_generator.citygraph_dataset import get_dataset_from_config
from transport.routes_generator.utils import get_eval_cfg
from transport.routes_generator.eval_route_generator import eval_model
from transport.routes_generator.torch_utils import dump_routes
from transport.routes_generator import bee_colony
import transport.routes_generator.utils as lrnu

In [20]:
# Root paths (adjust only here if needed)
ROOT_DIR = Path.cwd().parent.parent
DATA_DIR = ROOT_DIR / "examples/data"
CFG_DIR = ROOT_DIR / "transport/routes_generator/cfg"
MODEL_WEIGHTS_PATH = ROOT_DIR / "examples/data/model_weights/inductive_random_graphs_weighted_connectivity.pt"

# Verify paths
print("Root directory:", ROOT_DIR)
print("Config directory:", CFG_DIR)
print("Model weights exist:", MODEL_WEIGHTS_PATH.exists())

Root directory: /root/transport
Config directory: /root/transport/transport/routes_generator/cfg
Model weights exist: True


## 2. Load Benchmark Instances

### 2.1 Mandl's Benchmark (Classic 15-node instance)

In [21]:
# from pathlib import Path
# import pickle
# import geopandas as gpd
# import pandas as pd
# import numpy as np
# import torch
# import networkx as nx

# # =====================================================================================
# # 1. Пути
# # =====================================================================================

# ROOT_DIR = Path("/root/transport")
# DATA_DIR = ROOT_DIR / "examples" / "data" / "data"

# print("Root:", ROOT_DIR)
# print("Data:", DATA_DIR)

# # =====================================================================================
# # 2. Чтение файлов
# # =====================================================================================

# with open(DATA_DIR / "all_nodes.pkl", "rb") as f:
#     all_nodes = pickle.load(f)

# with open(DATA_DIR / "all_stops.pkl", "rb") as f:
#     all_stops = pickle.load(f)

# with open(DATA_DIR / "demand_matrix_all_nodes.pkl", "rb") as f:
#     demand_matrix_all_nodes = pickle.load(f)

# with open(DATA_DIR / "demand_matrix_stops.pkl", "rb") as f:
#     demand_matrix_stops = pickle.load(f)

# with open(DATA_DIR / "G_all_nodes.pkl", "rb") as f:
#     G_all_nodes = pickle.load(f)

# with open(DATA_DIR / "G_stops.pkl", "rb") as f:
#     G_stops = pickle.load(f)

# with open(DATA_DIR / "time_matrix_all_nodes.pkl", "rb") as f:
#     time_matrix_all_nodes = pickle.load(f)

# with open(DATA_DIR / "time_matrix_stops.pkl", "rb") as f:
#     time_matrix_stops = pickle.load(f)


# # =====================================================================================
# # 3. Предобработка: перевод матриц времени в float
# # =====================================================================================

# def clean_matrix(mat):
#     """Преобразуем строки в float, заменяем 'inf' -> np.inf."""
#     mat = np.array(mat)

#     # заменяем все строковые "inf" на np.inf
#     mat = np.where(mat == "inf", np.inf, mat)
#     mat = np.where(mat == "Inf", np.inf, mat)
#     mat = np.where(mat == "INF", np.inf, mat)

#     # заменяем все строковые "nan" на np.nan
#     mat = np.where(mat == "nan", np.nan, mat)
#     mat = np.where(mat == "NaN", np.nan, mat)

#     # теперь конвертируем всё в float
#     mat = mat.astype(float)

#     return mat


# time_matrix_all_nodes = clean_matrix(time_matrix_all_nodes)
# time_matrix_stops = clean_matrix(time_matrix_stops)


# # =====================================================================================
# # 4. Вывод структуры
# # =====================================================================================

# print("\n=== GEO DATA ===")
# print("all_nodes shape:", all_nodes.shape)
# print("all_stops shape:", all_stops.shape)

# print("\n=== MATRICES ===")
# print("demand_matrix_all_nodes:", demand_matrix_all_nodes.shape)
# print("demand_matrix_stops:", demand_matrix_stops.shape)
# print("time_matrix_all_nodes:", time_matrix_all_nodes.shape)
# print("time_matrix_stops:", time_matrix_stops.shape)

# print("\n=== GRAPHS ===")
# print("G_all_nodes:", G_all_nodes.number_of_nodes(), "nodes")
# print("G_stops:", G_stops.number_of_nodes(), "nodes")

# # =====================================================================================
# # 5. Функции формирования input_tensors
# # =====================================================================================

# def load_all_nodes_instance():
#     # сортируем по nodeID = порядок в матрицах
#     nodes_sorted = all_nodes.sort_values("nodeID")

#     demand = torch.tensor(demand_matrix_all_nodes.values, dtype=torch.float32)

#     node_locs = torch.tensor(
#         nodes_sorted[["x_coord", "y_coord"]].values,
#         dtype=torch.float32
#     )

#     street_adj = torch.tensor(time_matrix_all_nodes, dtype=torch.float32)
#     street_adj[street_adj == np.inf] = torch.inf

#     print(f"\nLoaded ALL_NODES instance: {len(nodes_sorted)} nodes")
#     return {
#         "street_adj": street_adj,
#         "demand": demand,
#         "node_locs": node_locs
#     }


# def load_stops_instance():
#     nodes_sorted = all_stops.sort_values("nodeID")

#     demand = torch.tensor(demand_matrix_stops.values, dtype=torch.float32)

#     node_locs = torch.tensor(
#         nodes_sorted[["x_coord", "y_coord"]].values,
#         dtype=torch.float32
#     )

#     street_adj = torch.tensor(time_matrix_stops, dtype=torch.float32)
#     street_adj[street_adj == np.inf] = torch.inf

#     print(f"\nLoaded STOPS instance: {len(nodes_sorted)} stops")
#     return {
#         "street_adj": street_adj,
#         "demand": demand,
#         "node_locs": node_locs
#     }


# # =====================================================================================
# # 6. Создаём два варианта входных данных
# # =====================================================================================

# input_tensors_all = load_all_nodes_instance()
# input_tensors_stops = load_stops_instance()


# print("\n=== READY ===")
# print("ALL NODES tensors:")
# print(" street_adj:", input_tensors_all["street_adj"].shape)
# print(" demand:", input_tensors_all["demand"].shape)
# print(" node_locs:", input_tensors_all["node_locs"].shape)

# print("\nSTOPS tensors:")
# print(" street_adj:", input_tensors_stops["street_adj"].shape)
# print(" demand:", input_tensors_stops["demand"].shape)
# print(" node_locs:", input_tensors_stops["node_locs"].shape)


In [22]:
def load_mandl_instance():
    data_dir = DATA_DIR / "banchmark"
    instance_name = "Mandl"

    # Coordinates
    coords_path = data_dir / f"{instance_name}Coords.txt"
    node_locs = torch.tensor(np.genfromtxt(coords_path, skip_header=1), dtype=torch.float32)

    # Travel times (in seconds)
    tt_path = data_dir / f"{instance_name}TravelTimes.txt"
    street_adj = torch.tensor(np.genfromtxt(tt_path), dtype=torch.float32) * 60

    # Demand matrix
    dmd_path = data_dir / f"{instance_name}Demand.txt"
    demand = torch.tensor(np.genfromtxt(dmd_path), dtype=torch.float32)

    print(f"Mandl instance loaded:")
    print(f"  Nodes: {node_locs.shape[0]}, Demand shape: {demand.shape}")

    return {
        'street_adj': street_adj,
        'demand': demand,
        'node_locs': node_locs
    }

### 2.2 Tartu Real-World Network (199 nodes)

In [23]:
def load_tartu_instance():
    data_dir = DATA_DIR / "tartu"

    # Load graph
    with open(data_dir / "bus_graph_Tartu.pkl", "rb") as f:
        G = pickle.load(f)

    # Load demand (OD matrix)
    OD = pd.read_csv(data_dir / "OD_matrix_Tartu.csv", index_col=0)
    demand = torch.tensor(OD.values, dtype=torch.float32)

    # Sort nodes to match OD matrix order
    nodes = sorted(G.nodes())
    node_to_idx = {node: i for i, node in enumerate(nodes)}

    # Node coordinates
    node_locs = torch.tensor([[G.nodes[n]['x'], G.nodes[n]['y']] for n in nodes], dtype=torch.float32)

    # Build symmetric travel time matrix
    n = len(nodes)
    inf = float('inf')
    adj_matrix = np.full((n, n), inf, dtype=np.float32)
    np.fill_diagonal(adj_matrix, 0.0)

    for u, v, data in G.edges(data=True):
        if 'time_min' in data:
            t = data['time_min']
            i, j = node_to_idx[u], node_to_idx[v]
            adj_matrix[i, j] = t
            adj_matrix[j, i] = t

    street_adj = torch.tensor(adj_matrix)
    street_adj[street_adj == inf] = torch.inf

    print(f"Tartu instance loaded: {n} nodes")

    return {
        'street_adj': street_adj,
        'demand': demand,
        'node_locs': node_locs
    }

In [24]:
# Uncomment to use Tartu instead
input_tensors = load_tartu_instance()
# input_tensors = load_mandl_instance()

Tartu instance loaded: 199 nodes


## 3. Evaluation Parameters

In [25]:
eval_params = {
    "dataset_name": "tensor",
    "n_routes": 5,
    "min_route_len": 10,
    "max_route_len": 20,
    "demand_time_weight": 0.33,
    "route_time_weight": 0.33,
    "median_connectivity_weight": 0.33,
    "run_name": "mandl_eval",
    "model_weights": str(MODEL_WEIGHTS_PATH),
    "add_detour_penalty": False
}

## 4. Method 1: Learning-based Construction (LC)

In [26]:
# Load config
cfg_lc = get_eval_cfg(
    cfg_dir=str(CFG_DIR),
    base_cfg_name="eval_model_mumford",
    params=eval_params
)

# Build dataset and dataloader
test_ds = get_dataset_from_config(cfg_lc.eval.dataset, tensors=input_tensors)
test_dl = DataLoader(test_ds, batch_size=cfg_lc.batch_size)

# Initialize model
DEVICE, run_name_lc, _, cost_obj, model_lc = lrnu.process_standard_experiment_cfg(
    cfg_lc, run_name_prefix="lc_", weights_required=True
)

print(f"Running LC on {DEVICE}...")

_, unserved_lc, metrics_lc, routes_lc = eval_model(
    model_lc,
    test_dl,
    cfg_lc.eval,
    cost_obj,
    n_samples=cfg_lc.get("n_samples", 1),
    return_routes=True,
    silent=True,
    device=DEVICE
)

dump_routes(run_name_lc, routes_lc.cpu())

print("\nLC Results")
print("-" * 50)
print(f"Cost: {metrics_lc['cost'].item():.4f}")
print(f"Unserved demand (%): {metrics_lc['$d_{un}$'].item():.2f}")
print(f"Average Travel Time (ATT): {metrics_lc['ATT'].item():.4f}")
print(f"Routes:\n{routes_lc}")

/root/transport/transport/routes_generator/utils.py:171: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(cfg.model.weights,


Running LC on cpu...

LC Results
--------------------------------------------------
Cost: 6.3086
Unserved demand (%): 85.16
Average Travel Time (ATT): 4.9219
Routes:
tensor([[[131, 132, 171, 181,  23,  22, 191,  20,  13,  14,  19,  12,  10,   7,
           11,   8,  79,  21,  16],
         [ 92,  98, 103, 176,  91,  90, 100,  89,  95, 109,  37,  33,  36,  34,
          167, 164, 165, 154, 187],
         [197,   2,  86,  17,  85,   7,  10,  12,  39,  38, 115,   5, 142, 144,
          136, 148, 120,  -1,  -1],
         [106, 104,  88,  97,  94,  89, 137, 135, 126, 130,  54,  65,  64,  60,
           62,   6,  -1,  -1,  -1],
         [113, 108,  99, 179, 183, 100,  89,  95, 109,  12,  10,   7,  83, 156,
          151, 161, 158,  -1,  -1]]])


/root/transport/transport/routes_generator/utils.py:319: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at ../aten/src/ATen/native/ReduceOps.cpp:1823.)
  out_stats = (final_costs.mean(), final_costs.std(), unserved_demand, all_metrics)


## 5. Method 2: Neural Bee Colony Optimization (Neural BCO)

In [ ]:
# Use same parameters but switch to BCO config
bco_params = eval_params.copy()
bco_params.update({
    "run_name": "neural_bco_mandl",
    "starting_routes_init": "tensor"  # Use LC routes as warm-start
})

cfg_bco = get_eval_cfg(
    cfg_dir=str(CFG_DIR),
    base_cfg_name="neural_bco_mumford",
    params=bco_params
)

# Re-use same dataset
test_dl_bco = DataLoader(test_ds, batch_size=cfg_bco.batch_size)

# Initialize bee colony (with optional neural guidance)
use_neural_bees = cfg_bco.get("neural_bees", True)
prefix = "neural_bco_" if use_neural_bees else "bco_"

DEVICE, run_name_bco, sum_writer, cost_obj_bco, bee_model = lrnu.process_standard_experiment_cfg(
    cfg_bco, prefix, weights_required=True
)

if bee_model is not None:
    bee_model.eval()

print(f"Running Neural BCO (warm-started from LC)...")

test_output = lrnu.test_method(
    bee_colony,
    test_dl_bco,
    cfg_bco.eval,
    cfg_bco.init,
    cost_obj_bco,
    n_bees=cfg_bco.n_bees,
    n_iterations=cfg_bco.n_iterations,
    device=DEVICE,
    bee_model=bee_model,
    return_routes=True,
    routes_tensor=routes_lc,  # Warm-start!
    silent=True
)

routes_bco = test_output[-1]
metrics_bco = test_output[-2]
unserved_bco = test_output[-3]

dump_routes(run_name_bco, routes_bco)

print("\nNeural BCO Results")
print("-" * 50)
print(f"Cost: {metrics_bco['cost'].item():.4f}")
print(f"Unserved demand (%): {metrics_bco['$d_{un}$'].item():.2f}")
print(f"Average Travel Time (ATT): {metrics_bco['ATT'].item():.4f}")
print(f"Improvement vs LC: {(metrics_lc['cost'] - metrics_bco['cost']).item():.4f}")
print(f"Final routes:\n{routes_bco}")

Running Neural BCO (warm-started from LC)...


In [ ]:
# Running Neural BCO (warm-started from LC)...

# Neural BCO Results
# --------------------------------------------------
# Cost: 0.7824
# Unserved demand (%): 1.67
# Average Travel Time (ATT): 15.0655
# Improvement vs LC: 0.0089
# Final routes:
# [[tensor([7, 5, 2, 1, 0]), tensor([ 8, 14,  6,  9, 10]), tensor([12, 10, 11]), tensor([ 4,  3,  5, 14]), tensor([14,  7]), tensor([12, 13])]]

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import warnings
warnings.filterwarnings("ignore")

# ========================================
# SETTINGS
# ========================================
pkl_files = [
    'neural_bco_exp_weighted_connectivity_mandl_pp_0_op_0_cp_1_generated.pkl',
    'neural_bco_exp_weighted_connectivity_mandl_pp_0_op_1_cp_0_generated.pkl',
    'neural_bco_exp_neuro_mandl_pp_0.9_op_0.1_cp_0.0_generated.pkl',
    'neural_bco_exp_weighted_connectivity_mandl_pp_1_op_0_cp_0_generated.pkl',
]

# --- Coordinates & travel-time matrix ---
coords_path = 'MandlCoords.txt'
with open(coords_path, 'r') as f:
    lines = [l.strip() for l in f.readlines() if l.strip()]
n_nodes = int(lines[0])
coords = np.array([list(map(float, line.split())) for line in lines[1:1+n_nodes]])
pos = {i: coords[i] for i in range(n_nodes)}

times_path = 'MandlTravelTimes.txt'
travel_times = []
with open(times_path, 'r') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        values = line.split()
        row = [float('inf') if v == 'Inf' else float(v) for v in values]
        while len(row) < n_nodes: row.append(float('inf'))
        row = row[:n_nodes]
        travel_times.append(row)
travel_times = np.array(travel_times, dtype=float)

# --- Background graph ---
G = nx.DiGraph()
for i in range(n_nodes):
    for j in range(n_nodes):
        if i != j and travel_times[i, j] < float('inf'):
            G.add_edge(i, j, weight=travel_times[i, j])

# --- Utility function for title/filename ---
def get_title_and_filename(fname):
    parts = fname.replace('.pkl', '').split('_')
    pp = parts[parts.index('pp') + 1]
    op = parts[parts.index('op') + 1]
    cp = parts[parts.index('cp') + 1]

    perspective = []
    if pp == '1': perspective.append('Passenger')
    if op == '1': perspective.append('Operator')
    if cp == '1': perspective.append('Connectivity')

    title = 'Route Set – ' + ' + '.join(perspective) if perspective else 'Route Set – None'
    short_name = '_'.join(perspective).lower().replace(' ', '_')
    if not short_name: short_name = 'none'
    return title, f'mandl_routes_{short_name}.png'

# ========================================
# PLOT: 3 subplots side by side, separate titles/legends
# ========================================
# ========================================
# PLOT: 4 subplots side by side, separate titles/legends
# ========================================
fig, axes = plt.subplots(1, 4, figsize=(34, 10), constrained_layout=True)  # ← 4 вместо 3
colors = plt.cm.Set1(np.linspace(0, 1, 10))
node_size = 300
offset_distance = 0.3

for fig_idx, pkl_path in enumerate(pkl_files):
    ax = axes[fig_idx]

    # --- Load routes ---
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    routes_tensor = data[0]
    print(f"Loaded {pkl_path}: {routes_tensor.shape[0]} routes")

    # --- Edge usage & order ---
    edge_order = {}
    for idx, route in enumerate(routes_tensor):
        route = [int(x) for x in route if x != -1]
        for k in range(len(route) - 1):
            u, v = route[k], route[k+1]
            edge_order.setdefault((u, v), []).append(idx)

    # --- Background edges ---
    for u, v, d in G.edges(data=True):
        x1, y1 = pos[u]; x2, y2 = pos[v]
        ax.plot([x1, x2], [y1, y2], color='lightgray', lw=1.2, alpha=0.6, zorder=1)

    # --- Edge weights ---
    for i in range(n_nodes):
        for j in range(i + 1, n_nodes):
            if travel_times[i, j] >= float('inf'): continue
            time = int(travel_times[i, j])
            x1, y1 = pos[i]; x2, y2 = pos[j]
            mx, my = (x1 + x2) / 2, (y1 + y2) / 2
            angle = np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi
            perp = angle + 90
            offset = 0.25
            dx = offset * np.cos(np.radians(perp))
            dy = offset * np.sin(np.radians(perp))
            ax.text(mx + dx, my + dy, f'{time}', fontsize=8, fontweight='bold',
                    color='darkred', ha='center', va='center',
                    bbox=dict(boxstyle="round,pad=0.2", facecolor='white',
                              edgecolor='red', linewidth=0.7, alpha=0.95),
                    rotation=angle, rotation_mode='anchor', zorder=3)

    # --- Nodes ---
    ax.scatter(coords[:, 0], coords[:, 1], c='black', s=node_size, zorder=5)
    for i, (x, y) in enumerate(coords):
        ax.text(x, y, str(i), fontsize=12, fontweight='bold', color='white',
                ha='center', va='center', zorder=6)

    # --- Routes ---
    handles = []
    for idx, route_tensor in enumerate(routes_tensor):
        route = [int(x) for x in route_tensor if x != -1]
        if len(route) < 2: continue
        color = colors[idx % len(colors)]
        total_time = int(sum(travel_times[route[k], route[k+1]] for k in range(len(route) - 1)))

        segments = []
        for k in range(len(route) - 1):
            u, v = route[k], route[k+1]
            x1, y1 = pos[u]; x2, y2 = pos[v]
            dx, dy = x2 - x1, y2 - y1
            length = np.hypot(dx, dy)
            if length == 0: continue
            ux, uy = dx / length, dy / length
            px, py = -uy, ux
            order_list = edge_order.get((u, v), [])
            pos_in_edge = order_list.index(idx) if idx in order_list else 0
            total = len(order_list)
            offset_factor = (pos_in_edge - (total - 1) / 2) if total > 0 else 0
            offset = offset_factor * offset_distance
            ox = px * offset
            oy = py * offset
            x1o, y1o = x1 + ox, y1 + oy
            x2o, y2o = x2 + ox, y2 + oy
            segments.append([(x1o, y1o), (x2o, y2o)])

        if segments:
            line_x = [p[0] for seg in segments for p in seg]
            line_y = [p[1] for seg in segments for p in seg]
            ax.plot(line_x, line_y, color=color, linewidth=4, alpha=0.9, zorder=4)

        # Arrows
        for k in range(len(route) - 1):
            u, v = route[k], route[k+1]
            x1, y1 = pos[u]; x2, y2 = pos[v]
            dx, dy = x2 - x1, y2 - y1
            length = np.hypot(dx, dy)
            if length == 0: continue
            ux, uy = dx / length, dy / length
            px, py = -uy, ux
            order_list = edge_order.get((u, v), [])
            pos_in_edge = order_list.index(idx) if idx in order_list else 0
            total = len(order_list)
            offset_factor = (pos_in_edge - (total - 1) / 2) if total > 0 else 0
            offset = offset_factor * offset_distance
            ox = px * offset
            oy = py * offset
            x1o, y1o = x1 + ox, y1 + oy
            x2o, y2o = x2 + ox, y2 + oy
            ax.annotate('', xy=(x2o, y2o), xytext=(x1o, y1o),
                        arrowprops=dict(arrowstyle='->', color=color, lw=2.2, alpha=0.8),
                        zorder=4)

        # Legend handle
        path_str = ' → '.join(map(str, route))
        label = f'Route {idx}: {path_str} ({total_time} min)'
        handles.append(plt.Line2D([0], [0], color=color, lw=5, label=label))

    # --- Title & Legend under plot ---
    title, _ = get_title_and_filename(pkl_path)
    ax.set_title(title, fontsize=16, fontweight='bold', pad=15)
    ax.set_aspect('equal')
    ax.axis('off')

    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.05),
              fontsize=10, ncol=1, frameon=True, fancybox=True, title='Routes', title_fontsize=11)

# --- Save the full figure with 3 plots ---
plt.savefig('mandl_routes_side_by_side.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
БЕЗ ДЕТУРА

Neural BCO Results
--------------------------------------------------
Cost: 0.3835
Unserved demand (%): 0.06
Average Travel Time (ATT): 14.1606
Improvement vs LC: 0.0201
Final routes:
[[tensor([ 0,  1,  2,  5,  7,  9, 10]), tensor([11,  3,  5, 14,  6]), tensor([13, 12, 10]), tensor([ 8, 14]), tensor([ 7, 14]), tensor([4, 3])]]
/root/transport/transport/routes_generator/utils.py:319: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at ../aten/src/ATen/native/ReduceOps.cpp:1823.)
  out_stats = (final_costs.mean(), final_costs.std(), unserved_demand, all_metrics)

In [ ]:

LC Results
--------------------------------------------------
Cost: 0.7913
Unserved demand (%): 0.00
Average Travel Time (ATT): 12.5106
Routes:
tensor([[[ 0,  1,  2,  5,  7,  9, 13],
         [ 8, 14,  6,  9, 10, -1, -1],
         [12, 10, 11, -1, -1, -1, -1],
         [ 4,  3,  5, 14, -1, -1, -1],
         [14,  7, -1, -1, -1, -1, -1],
         [12, 13, -1, -1, -1, -1, -1]]])
/root/transport/transport/routes_generator/utils.py:319: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at ../aten/src/ATen/native/ReduceOps.cpp:1823.)
  out_stats = (final_costs.mean(), final_costs.std(), unserved_demand, all_metrics)

## 6. Summary Comparison

In [ ]:
### 6. Summary Comparison (fixed)

print("=" * 60)
print("Final Comparison".center(60))
print("=" * 60)
print(f"{'Method':<15} {'Cost':<10} {'Unserved (%)':<15} {'ATT':<10} {'Time (s)':<10}")
print("-" * 60)

def _val(d, key, default=0.0):
    """Safely extract scalar from tensor or return default."""
    v = d.get(key, torch.tensor(default))
    return v.item() if hasattr(v, "item") else float(v)

# LC row
lc_cost   = _val(metrics_lc, 'cost')
lc_un     = _val(metrics_lc, '$d_{un}$')
lc_att    = _val(metrics_lc, 'ATT')
lc_time   = _val(metrics_lc, 'average wall-clock duration', 0)

# Neural BCO row
bco_cost  = _val(metrics_bco, 'cost')
bco_un    = _val(metrics_bco, '$d_{un}$')
bco_att   = _val(metrics_bco, 'ATT')
bco_time  = _val(metrics_bco, 'average wall-clock duration', 0)

print(f"{'LC':<15} {lc_cost:<10.4f} {lc_un:<15.2f} {lc_att:<10.4f} {lc_time:<10.1f}")
print(f"{'Neural BCO':<15} {bco_cost:<10.4f} {bco_un:<15.2f} {bco_att:<10.4f} {bco_time:<10.1f}")

# Optional: improvement highlight
if bco_cost < lc_cost:
    improvement = lc_cost - bco_cost
    print(f"\nNeural BCO improves cost by {improvement:.4f} ({improvement/lc_cost*100:.2f}% better)")